# 🎬 Audience Ratings Behaviour — Notebook 8
## Audience Behaviour Analysis

**Series:** May Newsletter — Movie Intelligence (Part 3 of 3)  
**Prerequisite:** Run Notebook 7 first

### Questions we answer
1. **Do popular movies receive better ratings?** (revenue vs audience score)
2. **Are profitable movies actually well-liked?** (ROI vs audience score)
3. **Which genres polarise audiences the most?** (rating variance by genre)
4. **Who are the hidden gems?** (low revenue, high rating)
5. **Who are the most overhyped?** (high revenue, low rating)
6. **How does TMDB critical score compare to user behaviour scores?** (vote_average vs avg_user_rating)


## 0 · Setup

In [1]:
MIDNIGHT  = "#1a1a2e"
GOLD      = "#e8b94f"
CRIMSON   = "#c0392b"
SILVER    = "#bdc3c7"
CREAM     = "#f5f5f0"
DARK_TEXT = "#2c2c2c"
TEAL      = "#1abc9c"
PURPLE    = "#8e44ad"

import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : CREAM,
    "axes.edgecolor"   : DARK_TEXT,
    "axes.labelcolor"  : DARK_TEXT,
    "axes.titlesize"   : 13,
    "axes.titleweight" : "bold",
    "axes.labelsize"   : 11,
    "xtick.color"      : DARK_TEXT,
    "ytick.color"      : DARK_TEXT,
    "xtick.labelsize"  : 9,
    "ytick.labelsize"  : 9,
    "grid.color"       : "#e0e0e0",
    "grid.linestyle"   : "--",
    "grid.linewidth"   : 0.6,
    "legend.fontsize"  : 9,
    "font.family"      : "DejaVu Sans",
})

def style_spines(ax, keep=("bottom", "left")):
    for spine in ax.spines.values():
        spine.set_visible(False)
    for s in keep:
        ax.spines[s].set_visible(True)
        ax.spines[s].set_color("#cccccc")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings("ignore")

df = pd.read_parquet("audience_clean.parquet")
print(f"Loaded: {df.shape[0]:,} films")
df[["title","revenue","ROI","avg_user_rating","rating_std","rating_count","quadrant"]].head(5)


Loaded: 1,657 films


,title,revenue,ROI,avg_user_rating,rating_std,rating_count,quadrant
0,Toy Story,373554033.0,11.451801,3.872470,0.958981,247,Blockbuster
1,Jumanji,262797249.0,3.043035,3.401869,0.880714,107,Overhyped
2,Waiting to Exhale,81452156.0,4.090760,2.384615,0.938835,13,Critical Failure
3,Heat,187436818.0,2.123947,3.884615,0.830928,104,Blockbuster
4,Sudden Death,64350171.0,0.838576,3.150000,0.812728,20,Critical Failure


## 1 · Revenue vs Audience Rating

In [2]:
r_rev,  p_rev  = pearsonr( df["revenue"],  df["avg_user_rating"])
r_roi,  p_roi  = spearmanr(df["ROI"],       df["avg_user_rating"])
r_pop,  p_pop  = pearsonr( df["popularity"],df["avg_user_rating"])

print("Correlations with Audience Rating:")
print(f"  Revenue    vs avg_user_rating : r  = {r_rev:.3f}  (p = {p_rev:.4f})")
print(f"  ROI        vs avg_user_rating : ρ  = {r_roi:.3f}  (p = {p_roi:.4f})")
print(f"  Popularity vs avg_user_rating : r  = {r_pop:.3f}  (p = {p_pop:.4f})")
print()
print("Interpretation:")
if abs(r_rev) < 0.2:
    print("  ▶ Weak revenue-rating link — commercial success does not predict audience satisfaction.")
elif abs(r_rev) < 0.4:
    print("  ▶ Moderate revenue-rating correlation — some connection but high variance.")
else:
    print("  ▶ Meaningful revenue-rating correlation.")


Correlations with Audience Rating:
  Revenue    vs avg_user_rating : r  = -0.015  (p = 0.5370)
  ROI        vs avg_user_rating : ρ  = 0.247  (p = 0.0000)
  Popularity vs avg_user_rating : r  = 0.122  (p = 0.0000)

Interpretation:
  ▶ Weak revenue-rating link — commercial success does not predict audience satisfaction.


## 2 · TMDB Vote Average vs User Rating Comparison

In [3]:
# vote_average is the TMDB community score (0-10); avg_user_rating is the MovieLens user score (0.5-5)
# Normalise both to 0–1 for fair comparison
df["vote_norm"]   = (df["vote_average"] - df["vote_average"].min()) / (df["vote_average"].max() - df["vote_average"].min())
df["rating_norm"] = (df["avg_user_rating"] - 0.5) / 4.5   # 0.5–5.0 scale

r_scores, _ = pearsonr(df["vote_norm"], df["rating_norm"])
df["score_gap"] = df["rating_norm"] - df["vote_norm"]

print(f"Correlation between TMDB vote and user rating (normalised) : r = {r_scores:.3f}")
print()
print("Score gap statistics (user rating − TMDB vote, normalised):")
print(df["score_gap"].describe().round(4).to_string())
print()
print("Films where user rating far exceeds TMDB vote (score_gap > 0.15):")
top_gap = df[df["score_gap"] > 0.15].nlargest(10, "score_gap")[["title","vote_average","avg_user_rating","score_gap","primary_genre"]]
print(top_gap.to_string(index=False))


Correlation between TMDB vote and user rating (normalised) : r = 0.842

Score gap statistics (user rating − TMDB vote, normalised):
count    1657.0000
mean       -0.0300
std         0.0742
min        -0.2670
25%        -0.0786
50%        -0.0323
75%         0.0135
max         0.3160

Films where user rating far exceeds TMDB vote (score_gap > 0.15):
                   title  vote_average  avg_user_rating  score_gap primary_genre
               Tom Jones           6.1         4.458333   0.315993     Adventure
             The Phantom           4.7         3.111111   0.271156     Adventure
             Zero Effect           6.0         4.125000   0.260101        Comedy
        An Ideal Husband           6.3         4.250000   0.233333         Drama
The Crow: City of Angels           5.2         3.272727   0.216162        Action
               RoboCop 3           4.2         2.395833   0.203114        Action
       Romeo Is Bleeding           5.7         3.600000   0.197980        Action
 

## 3 · Genre Polarisation Analysis

In [4]:
MIN_FILMS = 20

genre_audience = (
    df.groupby("primary_genre")
    .agg(
        film_count     = ("title",           "count"),
        avg_rating     = ("avg_user_rating",  "mean"),
        median_rating  = ("avg_user_rating",  "median"),
        avg_std        = ("rating_std",        "mean"),
        pct_divisive   = ("polarisation",      lambda x: (x == "Divisive").mean() * 100),
        avg_revenue_M  = ("revenue",           lambda x: x.mean() / 1e6),
        avg_ROI        = ("ROI",               "median"),
    )
    .query(f"film_count >= {MIN_FILMS}")
    .sort_values("avg_std", ascending=False)
    .reset_index()
)

print(f"Genres with ≥{MIN_FILMS} films in ratings universe: {len(genre_audience)}")
print()
print("=== Most Polarising Genres (highest avg rating std dev) ===")
print(genre_audience.head(10)[["primary_genre","film_count","avg_rating","avg_std","pct_divisive"]].to_string(index=False))
print()
print("=== Most Consensus Genres (lowest avg rating std dev) ===")
print(genre_audience.tail(8)[["primary_genre","film_count","avg_rating","avg_std","pct_divisive"]].to_string(index=False))


Genres with ≥20 films in ratings universe: 11

=== Most Polarising Genres (highest avg rating std dev) ===
  primary_genre  film_count  avg_rating  avg_std  pct_divisive
         Horror          74    3.283877 0.976366     31.081081
        Fantasy          57    3.333635 0.971833     35.087719
Science Fiction          60    3.279373 0.970206     31.666667
         Comedy         319    3.375615 0.967962     32.601881
      Adventure         194    3.411931 0.944667     24.226804
        Romance          27    3.419181 0.942649     25.925926
      Animation          48    3.563478 0.939614     22.916667
         Action         275    3.286172 0.935961     23.636364
          Drama         378    3.668574 0.907680     20.370370
       Thriller          57    3.351900 0.899400     26.315789

=== Most Consensus Genres (lowest avg rating std dev) ===
primary_genre  film_count  avg_rating  avg_std  pct_divisive
       Comedy         319    3.375615 0.967962     32.601881
    Adventure      

## 4 · Quadrant Analysis — The Four Film Archetypes

In [5]:
quad_stats = (
    df.groupby("quadrant")
    .agg(
        film_count    = ("title",            "count"),
        avg_revenue_M = ("revenue",           lambda x: x.mean() / 1e6),
        avg_ROI       = ("ROI",               "median"),
        avg_rating    = ("avg_user_rating",   "mean"),
        avg_std       = ("rating_std",        "mean"),
        pct_divisive  = ("polarisation",      lambda x: (x == "Divisive").mean() * 100),
    )
    .reset_index()
)

print("Quadrant summary:")
print(quad_stats.round(2).to_string(index=False))


Quadrant summary:
        quadrant  film_count  avg_revenue_M  avg_ROI  avg_rating  avg_std  pct_divisive
     Blockbuster         345         327.12     4.62        3.83     0.87         10.14
Critical Failure         343          38.34     0.57        3.06     1.00         39.65
      Hidden Gem         485          31.16     1.89        3.88     0.87         13.20
       Overhyped         484         261.84     2.43        3.04     1.00         37.19


## 5 · Hidden Gems Leaderboard

In [6]:
# Hidden Gems: bottom 40% revenue, top 40% user rating, ≥10 ratings
rev_40  = df["revenue"].quantile(0.40)
rat_60  = df["avg_user_rating"].quantile(0.60)

hidden_gems = (
    df[(df["revenue"] <= rev_40) &
       (df["avg_user_rating"] >= rat_60) &
       (df["rating_count"] >= 10)]
    .sort_values("avg_user_rating", ascending=False)
)

print(f"Hidden Gems identified : {len(hidden_gems)}")
print()
print("=== Top 20 Hidden Gems ===")
cols = ["title","release_year","primary_genre","revenue","avg_user_rating","rating_count","ROI"]
print(hidden_gems.head(20)[cols].to_string(index=False))


Hidden Gems identified : 328

=== Top 20 Hidden Gems ===
                      title  release_year primary_genre    revenue  avg_user_rating  rating_count           ROI
The Best Years of Our Lives        1946.0         Drama 23650000.0         4.636364            11  1.026190e+01
   The Shawshank Redemption        1994.0         Drama 28341469.0         4.487138           311  1.336588e-01
                  Tom Jones        1963.0     Adventure 37600000.0         4.458333            12  3.660000e+01
          On the Waterfront        1954.0         Crime  9600000.0         4.448276            29  9.549451e+00
              All About Eve        1950.0         Drama    63463.0         4.434211            38 -9.546693e-01
                        Ran        1985.0        Action  4069653.0         4.423077            26 -6.461171e-01
          The African Queen        1951.0     Adventure 10750000.0         4.420000            50  7.269231e+00
                 Roger & Me        1989.0   Doc

## 6 · Most Overhyped Films

In [7]:
# Overhyped: top 40% revenue, bottom 40% user rating, ≥10 ratings
rev_60  = df["revenue"].quantile(0.60)
rat_40  = df["avg_user_rating"].quantile(0.40)

overhyped = (
    df[(df["revenue"] >= rev_60) &
       (df["avg_user_rating"] <= rat_40) &
       (df["rating_count"] >= 10)]
    .sort_values("comm_vs_audience_gap", ascending=False)
)

print(f"Overhyped films identified : {len(overhyped)}")
print()
print("=== Top 20 Most Overhyped ===")
print(overhyped.head(20)[cols].to_string(index=False))


Overhyped films identified : 307

=== Top 20 Most Overhyped ===
                                             title  release_year   primary_genre      revenue  avg_user_rating  rating_count       ROI
                                              2012        2009.0          Action  769653595.0         2.411765            17  2.848268
       Pirates of the Caribbean: On Stranger Tides        2011.0       Adventure 1045713802.0         2.650000            10  1.751878
Indiana Jones and the Kingdom of the Crystal Skull        2008.0       Adventure  786636033.0         2.562500            24  3.252087
               Transformers: Revenge of the Fallen        2009.0 Science Fiction  836297228.0         2.625000            12  4.575315
                                      Spider-Man 3        2007.0         Fantasy  890871626.0         2.717391            23  2.452991
                                          Godzilla        1998.0 Science Fiction  379014294.0         2.178571            28  

## 7 · Rating Behaviour Over Time

In [8]:
# Has the gap between commercial success and audience approval shifted by decade?
decade_stats = (
    df[df["decade"].between(1970, 2017)]
    .groupby("decade")
    .agg(
        film_count        = ("title",                "count"),
        avg_rating        = ("avg_user_rating",       "mean"),
        avg_vote          = ("vote_average",           "mean"),
        avg_std           = ("rating_std",             "mean"),
        avg_gap           = ("comm_vs_audience_gap",   "mean"),
        pct_hidden_gem    = ("quadrant",               lambda x: (x == "Hidden Gem").mean() * 100),
        pct_overhyped     = ("quadrant",               lambda x: (x == "Overhyped").mean() * 100),
    )
    .reset_index()
)

print("Decade-level audience behaviour trends:")
print(decade_stats.round(2).to_string(index=False))


Decade-level audience behaviour trends:
 decade  film_count  avg_rating  avg_vote  avg_std  avg_gap  pct_hidden_gem  pct_overhyped
   1970          82        3.73      7.21     0.89   -24.39           43.90          14.63
   1980         230        3.53      6.87     0.90   -14.61           43.04          12.17
   1990         554        3.32      6.53     0.96     4.34           23.10          32.85
   2000         530        3.36      6.68     0.95    15.49           18.49          41.51
   2010         129        3.63      7.13     0.91    15.14           13.18          31.01


## 8 · Key Facts for the Newsletter

In [9]:
print("=== Key Facts for Newsletter ===")
print(f"  Total films analysed        : {len(df):,}")
print(f"  Total user ratings          : {df['rating_count'].sum():,}")
print()
print(f"  Revenue vs rating r         : {r_rev:.3f}  ({'weak' if abs(r_rev)<0.2 else 'moderate' if abs(r_rev)<0.4 else 'strong'})")
print(f"  ROI vs rating ρ             : {r_roi:.3f}")
print()
print(f"  Most polarising genre       : {genre_audience.iloc[0]['primary_genre']}  (avg std = {genre_audience.iloc[0]['avg_std']:.3f})")
print(f"  Most consensus genre        : {genre_audience.iloc[-1]['primary_genre']} (avg std = {genre_audience.iloc[-1]['avg_std']:.3f})")
print()
print(f"  Hidden gems identified      : {len(hidden_gems):,}")
print(f"  Overhyped films identified  : {len(overhyped):,}")
print()

best_gem  = hidden_gems.iloc[0]
worst_hyp = overhyped.iloc[0]
print(f"  Best hidden gem             : {best_gem['title']} ({int(best_gem['release_year'])}) — "
      f"${best_gem['revenue']/1e6:.1f}M revenue, {best_gem['avg_user_rating']:.2f} rating")
print(f"  Most overhyped              : {worst_hyp['title']} ({int(worst_hyp['release_year'])}) — "
      f"${worst_hyp['revenue']/1e6:.0f}M revenue, {worst_hyp['avg_user_rating']:.2f} rating")
print()
print("Proceed to Notebook 9 → Newsletter Visualisations ▶")


=== Key Facts for Newsletter ===
  Total films analysed        : 1,657
  Total user ratings          : 69,592

  Revenue vs rating r         : -0.015  (weak)
  ROI vs rating ρ             : 0.247

  Most polarising genre       : Horror  (avg std = 0.976)
  Most consensus genre        : Crime (avg std = 0.873)

  Hidden gems identified      : 328
  Overhyped films identified  : 307

  Best hidden gem             : The Best Years of Our Lives (1946) — $23.6M revenue, 4.64 rating
  Most overhyped              : 2012 (2009) — $770M revenue, 2.41 rating

Proceed to Notebook 9 → Newsletter Visualisations ▶
